In [ ]:
# import python libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt # visualizing data
%matplotlib inline
import seaborn as sns

In [ ]:
# import csv file
df = pd.read_csv('Diwali Sales Data.csv', encoding= 'unicode_escape')

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
#drop unrelated/blank columns
df.drop(['Status', 'unnamed1'], axis=1, inplace=True)

In [ ]:
#check for null values
pd.isnull(df).sum()

In [ ]:
# Keep rows with the two numeric measures needed for sales summaries.
# Preserve missing demographic fields and label them as Unknown for segment analysis.
rows_before = len(df)
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')
df['Orders'] = pd.to_numeric(df['Orders'], errors='coerce')
df = df.dropna(subset=['Amount', 'Orders']).copy()
print(f'Rows before measure validation: {rows_before:,}')
print(f'Rows retained for sales analysis: {len(df):,}')
print(f'Rows excluded for missing/invalid Amount or Orders: {rows_before - len(df):,}')
for column in ['Gender', 'Age Group', 'Marital_Status', 'State', 'Occupation', 'Product_Category']:
    if column in df.columns:
        df[column] = df[column].fillna('Unknown').astype(str)

In [ ]:
# change data type
df['Amount'] = df['Amount'].astype('int')

In [ ]:
df['Amount'].dtypes

In [ ]:
df.columns

In [ ]:
# Check the cleaned column names and confirm the demographic field is present.
df[['Marital_Status']].value_counts(dropna=False).rename('record_count').reset_index()

In [ ]:
# describe() method returns description of the data in the DataFrame (i.e. count, mean, std, etc)
df.describe()

In [ ]:
# use describe() for specific columns
df[['Age', 'Orders', 'Amount']].describe()

## Business questions

- Which gender and age groups account for the largest recorded order volume and Amount?
- Which states rank highest by order count versus Amount?
- How does the recorded sales mix vary by marital status and occupation?
- Which product categories lead by frequency and by Amount?
- Which products appear most often by summed Orders?

## Data-quality decisions

The notebook checks missing values before analysis and excludes rows only when the numeric measures Amount or Orders are missing or invalid. Missing demographic/category values are retained as Unknown rather than silently removed. This keeps the exclusion rule explicit; rerun the notebook to verify the retained row count and rankings against the committed CSV. Segment patterns are descriptive, not causal.

# Exploratory Data Analysis

### Gender

In [ ]:
# plotting a bar chart for Gender and it's count

ax = sns.countplot(x = 'Gender',data = df)

for bars in ax.containers:
    ax.bar_label(bars)

In [ ]:
# plotting a bar chart for gender vs total amount

sales_gen = df.groupby(['Gender'], as_index=False)['Amount'].sum().sort_values(by='Amount', ascending=False)

ax=sns.barplot(x = 'Gender',y= 'Amount' ,data = sales_gen)
# Format y-axis labels
import matplotlib.ticker as mtick
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.0f'))

for bars in ax.containers:
    ax.bar_label(bars, fmt='%.0f')

The charts compare buyer-record counts with total recorded Amount by gender. A higher total amount does not by itself prove higher individual purchasing power; compare per-customer/order metrics if that is the business question.

### Age

In [ ]:
ax = sns.countplot(data = df, x = 'Age Group', hue = 'Gender')

for bars in ax.containers:
    ax.bar_label(bars)

In [ ]:
# Total Amount vs Age Group
sales_age = df.groupby(['Age Group'], as_index=False)['Amount'].sum().sort_values(by='Amount', ascending=False)

ax=sns.barplot(x = 'Age Group',y= 'Amount' ,data = sales_age)
# Format y-axis labels
import matplotlib.ticker as mtick
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.0f'))

for bars in ax.containers:
    ax.bar_label(bars, fmt='%.0f')

The 26–35 age band appears prominent in the plotted dataset. Validate this using both customer counts and total Amount, and avoid combining age and gender claims unless the grouped result directly supports them.

### State

In [ ]:
# total number of orders from top 10 states

sales_state = df.groupby(['State'], as_index=False)['Orders'].sum().sort_values(by='Orders', ascending=False).head(10)

sns.set(rc={'figure.figsize':(15,5)})
sns.barplot(data = sales_state, x = 'State',y= 'Orders')

In [ ]:
# total amount/sales from top 10 states

sales_state = df.groupby(['State'], as_index=False)['Amount'].sum().sort_values(by='Amount', ascending=False).head(10)

sns.set(rc={'figure.figsize':(15,5)})
sns.barplot(data = sales_state, x = 'State',y= 'Amount')

The charts rank states separately by summed Orders and summed Amount. A state can rank differently on volume and value; confirm the metric before making a sales-performance claim.

### Marital Status

In [ ]:
ax = sns.countplot(data = df, x = 'Marital_Status')

sns.set(rc={'figure.figsize':(7,5)})
for bars in ax.containers:
    ax.bar_label(bars)

In [ ]:
sales_state = df.groupby(['Marital_Status', 'Gender'], as_index=False)['Amount'].sum().sort_values(by='Amount', ascending=False)

sns.set(rc={'figure.figsize':(6,5)})
sns.barplot(data = sales_state, x = 'Marital_Status',y= 'Amount', hue='Gender')

Compare the recorded buyer counts and Amount by marital status and gender. These are descriptive associations in the sample and should not be interpreted as evidence that marital status causes spending differences.

### Occupation

In [ ]:
sns.set(rc={'figure.figsize':(20,5)})
ax = sns.countplot(data = df, x = 'Occupation')

for bars in ax.containers:
    ax.bar_label(bars)

In [ ]:
sales_state = df.groupby(['Occupation'], as_index=False)['Amount'].sum().sort_values(by='Amount', ascending=False)

sns.set(rc={'figure.figsize':(20,5)})
sns.barplot(data = sales_state, x = 'Occupation',y= 'Amount')

The occupation charts compare record counts and total Amount. The top occupation by volume need not be the top occupation by revenue; read each chart separately.

### Product Category

In [ ]:
sns.set(rc={'figure.figsize':(20,5)})
ax = sns.countplot(data = df, x = 'Product_Category')

for bars in ax.containers:
    ax.bar_label(bars)

In [ ]:
sales_state = df.groupby(['Product_Category'], as_index=False)['Amount'].sum().sort_values(by='Amount', ascending=False).head(10)

sns.set(rc={'figure.figsize':(20,5)})
sns.barplot(data = sales_state, x = 'Product_Category',y= 'Amount')

The category charts show both record frequency and total Amount. Use the revenue chart for value ranking and the count chart for frequency; they answer different questions.

In [ ]:
sales_state = df.groupby(['Product_ID'], as_index=False)['Orders'].sum().sort_values(by='Orders', ascending=False).head(10)

sns.set(rc={'figure.figsize':(20,5)})
sns.barplot(data = sales_state, x = 'Product_ID',y= 'Orders')

In [ ]:
# top 10 most sold products (same thing as above)

fig1, ax1 = plt.subplots(figsize=(12,7))
df.groupby('Product_ID')['Orders'].sum().nlargest(10).sort_values(ascending=False).plot(kind='bar')

## Conclusion:

###

### Conclusion

This EDA explores the festival-period dataset across gender, age group, state, marital status, occupation and product category. Use the separate count and Amount summaries to distinguish customer/order volume from recorded sales value. These patterns can help form campaign and assortment hypotheses, but they do not establish causation or guarantee future demand.

Thank you!